# Notebook 02 — Clean Data and Filter (Path A)

**Goal:** Take raw data, fix problems, keep **recent international matches (2021 onwards, all tournaments)**, and save a clean copy.

**Why 2021+?** Recent squads, coaches, and team form matter more than matches from the 1950s.

**Pipeline steps covered:**
1. Clean data
2. Handle missing values

**Important rule:** We never edit `data/raw/results.csv`. We save cleaned data to `data/processed/`.

# Step 1 — Load the Raw Data Again

We always start cleaning from the **original raw file**.
If we make a mistake, we can redo this step without losing anything.

In [8]:
import pandas as pd
from pathlib import Path

# Paths: raw = original file, processed = cleaned output folder
PROJECT_ROOT = Path("..")
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "results.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_PATH = PROCESSED_DIR / "matches_2021_onwards.csv"

# Load raw data into a DataFrame
df = pd.read_csv(RAW_PATH)

print("Raw data shape:", df.shape)

Raw data shape: (49477, 9)


# Step 2 — Check for Missing Values

**Missing values** = empty cells in the dataset.

`.isnull().sum()` counts how many empty cells exist in each column.

If a match has no score, we cannot know who won — so those rows are useless for our project.

In [9]:
# isnull() marks empty cells as True; .sum() counts them per column
missing_counts = df.isnull().sum()

print("Missing values per column:")
print(missing_counts)

# Show rows where home_score is missing (so we see what they look like)
print("\nExample rows with missing scores:")
df[df["home_score"].isnull()].head()

Missing values per column:
date           0
home_team      0
away_team      0
home_score    44
away_score    44
tournament     0
city           0
country        0
neutral        0
dtype: int64

Example rows with missing scores:


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49433,2026-06-19,Scotland,Morocco,NaN,NaN,FIFA World Cup,Foxborough,United States,True
49434,2026-06-19,Brazil,Haiti,NaN,NaN,FIFA World Cup,Philadelphia,United States,True
49435,2026-06-19,United States,Australia,NaN,NaN,FIFA World Cup,Seattle,United States,False
49436,2026-06-19,Turkey,Paraguay,NaN,NaN,FIFA World Cup,Santa Clara,United States,True
49437,2026-06-20,Germany,Ivory Coast,NaN,NaN,FIFA World Cup,Toronto,Canada,True


# Step 3 — Handle Missing Values

**Decision:** Drop rows where `home_score` or `away_score` is missing.

Why drop (not fill)?
- We cannot guess a final score
- Only 44 rows out of 49,477 — losing them is safe

`.dropna(subset=[...])` removes rows with empty values in the listed columns.

In [10]:
# Count before cleaning
rows_before = len(df)

# dropna removes rows where listed columns have missing values
df_clean = df.dropna(subset=["home_score", "away_score"]).copy()

rows_after = len(df_clean)

print(f"Rows before: {rows_before}")
print(f"Rows after:  {rows_after}")
print(f"Rows removed: {rows_before - rows_after}")

Rows before: 49477
Rows after:  49433
Rows removed: 44


# Step 4 — Convert Dates and Check Range

**Path A decision:** Use all international tournaments from **2021 onwards**.

This includes friendlies, World Cup qualifiers, Nations League, and continental cups.

First we convert `date` from text to a real date so we can filter by year.

In [11]:
# pd.to_datetime() converts the date column from text to real dates
df_clean["date"] = pd.to_datetime(df_clean["date"])

print("Full dataset date range:")
print("  Earliest:", df_clean["date"].min())
print("  Latest:  ", df_clean["date"].max())
print("  Total matches:", len(df_clean))

Full dataset date range:
  Earliest: 1872-11-30 00:00:00
  Latest:   2026-06-18 00:00:00
  Total matches: 49433


# Step 5 — Filter to 2021 Onwards (All Tournaments)

We keep every match where `date >= 2021-01-01`.

**NOT** FIFA World Cup finals only — that was the old version of this notebook.

We should end up with about **5,700+** matches.

In [12]:
# Filter: keep rows where date is on or after 2021-01-01
CUTOFF_DATE = "2021-01-01"
matches_df = df_clean[df_clean["date"] >= CUTOFF_DATE].copy()

print("Matches from 2021 onwards:", len(matches_df))
print("\nTop tournaments in our filtered data:")
print(matches_df["tournament"].value_counts().head(10))
print("\nSample rows:")
matches_df.head()

Matches from 2021 onwards: 5711

Top tournaments in our filtered data:
tournament
FIFA World Cup qualification            1610
Friendly                                1488
UEFA Nations League                      354
African Cup of Nations qualification     341
CONCACAF Nations League                  320
UEFA Euro qualification                  239
African Cup of Nations                   156
AFC Asian Cup qualification              105
UEFA Euro                                102
Gold Cup                                  93
Name: count, dtype: int64

Sample rows:


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
43722,2021-01-12,United Arab Emirates,Iraq,0.0,0.0,Friendly,Dubai,United Arab Emirates,False
43723,2021-01-18,Kuwait,Palestine,0.0,1.0,Friendly,Kuwait City,Kuwait,False
43724,2021-01-19,Dominican Republic,Puerto Rico,0.0,1.0,Friendly,Santo Domingo,Dominican Republic,False
43725,2021-01-22,Guatemala,Puerto Rico,1.0,0.0,Friendly,Guatemala City,Guatemala,False
43726,2021-01-25,Dominican Republic,Serbia,0.0,0.0,Friendly,Santo Domingo,Dominican Republic,False


# Step 6 — Quick Quality Checks

Before saving, we verify the cleaned data makes sense.

Checks:
- No missing scores
- No negative scores
- Date range starts at 2021

In [13]:
# Check 1: no missing scores
print("Missing scores:", matches_df[["home_score", "away_score"]].isnull().sum().sum())

# Check 2: no negative scores
print("Negative home scores:", (matches_df["home_score"] < 0).sum())
print("Negative away scores:", (matches_df["away_score"] < 0).sum())

# Check 3: date range (should start at 2021)
print("\nEarliest match:", matches_df["date"].min())
print("Latest match:", matches_df["date"].max())

# Check 4: unique teams count
all_teams = pd.concat([matches_df["home_team"], matches_df["away_team"]]).unique()
print("\nUnique teams:", len(all_teams))

Missing scores: 0
Negative home scores: 0
Negative away scores: 0

Earliest match: 2021-01-12 00:00:00
Latest match: 2026-06-18 00:00:00

Unique teams: 262


# Step 7 — Save Clean Data to processed/

`.to_csv()` saves a DataFrame to a CSV file.

`index=False` means "don't save row numbers as an extra column."

From now on, we use **`data/processed/matches_2021_onwards.csv`** for modeling — not the raw file.

In [14]:
# Create processed folder if it does not exist yet
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Save cleaned 2021+ match data
matches_df.to_csv(PROCESSED_PATH, index=False)

print("Saved to:", PROCESSED_PATH.resolve())
print("File exists:", PROCESSED_PATH.exists())
print("Rows saved:", len(matches_df))

Saved to: C:\Users\HP-15\Desktop\worldcup_predictor\data\processed\matches_2021_onwards.csv
File exists: True
Rows saved: 5711


# Summary

| Step | Result |
|------|--------|
| Started with | ~49,477 raw matches |
| Removed missing scores | 44 rows dropped |
| Filtered to | **2021 onwards, all tournaments** (~5,700+ matches) |
| Saved to | `data/processed/matches_2021_onwards.csv` |

**Path A:** We will add coach, formation, and player stats in a later phase.

**Next notebook:** Feature engineering — create our target column (Win/Draw/Loss) and useful features.